# Normalization pipeline, step by step

This notebook runs `src/pipeline/build.py` one step at a time, so each step can be shown and explained.
For a normal rebuild just run, from the project root:

```
python -m src.pipeline.build
```

| Step | Module | What it does |
|---|---|---|
| 1. adapters | `src/pipeline/adapters/` | one loader per chain: raw columns → common columns (and fixes known collector bugs) |
| 2. standardize | `src/pipeline/standardize.py`, `units.py`, `text.py` | units, pack sizes, € per kg/l/unit, clean names, brands |
| 3. quality | `standardize.py`, `build.py` | flags instead of deleting rows |
| 4. classify | `src/pipeline/classify.py` | product type per product, by text similarity to the taxonomy |
| 5. store | `src/pipeline/db.py` | SQLite `data/supermarkets.db` + `data/products_normalized.csv` |

In [2]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "pipeline").exists())
sys.path.insert(0, str(ROOT))

import pandas as pd
from src.pipeline import config
from src.pipeline.build import prepare, build, format_report
from src.pipeline.classify import decide, find_similar
from src.pipeline.evaluate import evaluate, load_labels, sample_for_labelling

## Steps 1–3: load every chain, standardize, first flags

`prepare()` runs the adapters, `standardize()` and picks the vectorizer (cached embeddings if available,
otherwise TF-IDF). Nothing is dropped: a product without a price per unit gets `flag_missing_ppu`.

In [3]:
p = prepare()
print("Vectorizer:", p["report"]["method"], *p["report"]["warnings"], sep="\n")
print(len(p["products"]), "products,", len(p["hits"]), "search hits")
p["observations"][["price_per_unit", "unit", "ppu_source", "flag_missing_ppu"]].describe(include="all")

Vectorizer:
tfidf
13253 products, 12295 search hits


,price_per_unit,unit,ppu_source,flag_missing_ppu
count,13078.000000,13078,13078,13253
unique,NaN,11,2,2
top,NaN,KILO,site,False
freq,NaN,8444,12925,13078
mean,15.907452,NaN,NaN,NaN
std,168.063787,NaN,NaN,NaN
min,0.003000,NaN,NaN,NaN
25%,2.400000,NaN,NaN,NaN
50%,6.041000,NaN,NaN,NaN
75%,13.000000,NaN,NaN,NaN


In [4]:
# One product per chain, after standardization
p["products"].groupby("chain_id").head(1)[["product_key", "name", "name_clean", "brand", "is_private_label",
                                            "pack_count", "unit_quantity", "quantity_unit"]]

,product_key,name,name_clean,brand,is_private_label,pack_count,unit_quantity,quantity_unit
0,dia:504P6,Leche semidesnatada Dia Láctea pack 6 x 1 L,leche semidesnatada lactea pack,Dia,True,6.0,1.0,LITRO
918,mercadona:4241,"Aceite de oliva 0,4º Hacendado",aceite de oliva 0 4o,Hacendado,True,1.0,5.0,LITRO
5206,aldi:872200,Leche entera sin lactosa,leche entera sin lactosa,Milsani,True,1.0,1.0,LITRO
6067,alcampo:54180,AUCHAN Leche semidesnatada de vaca 6 x 1l Prod...,leche semidesnatada de vaca,Alcampo,True,6.0,1.0,LITRO
6763,ahorramas:70865,Leche Alipende 1l semidesnatada,leche semidesnatada,Alipende,True,1.0,1.0,LITRO
7156,bm:76411,Leche entera 1 l,leche entera,Bm,True,1.0,1.0,LITRO
10300,consum:7080604,Leche Semidesnatada Brik,leche semidesnatada brik,Consum,True,1.0,1.0,LITRO


## Step 4: classify

Each product is compared only with the types of the search terms that returned it
(`data/taxonomy/product_types.csv`). The closest one wins, unless it is a "noise" description,
the chain's category is excluded (pets, cosmetics…), or the score is below τ → `no_relevante`.

In [5]:
thresholds = config.THRESHOLDS[p["vectorizer"].method]
classification = decide(p["products"], p["hits"], p["taxonomy"], p["vectorizer"], thresholds)
print("tau/delta:", thresholds)
classification["method"].value_counts()

tau/delta: {'tau': 0.15, 'delta': 0.02}


method
tfidf                 9810
unclassified          2431
category_exclusion    1012
Name: count, dtype: int64

In [6]:
# Examples per type: does it look right?
examples = classification.merge(p["products"][["product_key", "name", "chain_id"]], on="product_key")
examples[examples["type_id"].isin(["leche_semidesnatada", "no_relevante"])].groupby("type_id").head(8)[
    ["chain_id", "name", "type_id", "score", "runner_up_type_id"]]

,chain_id,name,type_id,score,runner_up_type_id
0,dia,Leche semidesnatada Dia Láctea pack 6 x 1 L,leche_semidesnatada,0.794002,leche_desnatada
3,dia,Leche semidesnatada sin lactosa Dia Láctea pac...,leche_semidesnatada,0.645069,leche_sin_lactosa
4,dia,Leche semidesnatada Dia Láctea 1 L,leche_semidesnatada,0.840426,leche_desnatada
8,dia,Leche semidesnatada sin lactosa Dia Láctea 1 L,leche_semidesnatada,0.669231,leche_sin_lactosa
10,dia,Leche semidesnatada Asturiana pack 6 x 1 L,leche_semidesnatada,0.706348,leche_desnatada
15,dia,Leche semidesnatada Dia Láctea botella 1.5 L,leche_semidesnatada,0.745971,leche_desnatada
18,dia,Batido de cacao 90% leche Dia Láctea 6 x 200 ml,no_relevante,0.262958,leche_sin_lactosa
19,dia,Leche semidesnatada sin lactosa Asturiana pack...,leche_semidesnatada,0.624141,leche_sin_lactosa
20,dia,Leche semidesnatada Asturiana pack 6 x 1.5 L,leche_semidesnatada,0.706348,leche_desnatada
27,dia,Nata para cocinar textura espesa Dia Láctea 50...,no_relevante,0.100517,leche_sin_lactosa


## Step 5: build the database and the CSV

`build()` repeats the steps above and also adds `flag_outlier` (price per unit far from its type's usual range)
and `flag_unit_mismatch`, then writes `data/supermarkets.db` and `data/products_normalized.csv`.

In [7]:
print(format_report(build()))

Classifier: tfidf (tfidf-char-3-5)
chain        rows  products  %€/unit  %no_rel %low_conf  error
dia          1161       918    100.0     48.6       3.5
mercadona    4621      4288    100.0     71.1       2.6
aldi         1076       861     91.2     54.5       3.1
alcampo       711       696     96.3     15.5       6.9
ahorramas     398       393     82.4     18.6       6.1
bm           4076      3144     99.9     56.8       2.3
consum       3873      2953    100.0     55.0       2.5


## Hand labelling (≈ 100 products, 30–40 min)

The cell below shows 100 products (5 per search term, spread across chains), each with the classifier's guess.
For each one type:

- **Enter** → the guess is right
- **a number** → pick that type from the list
- **n** → `no_relevante` (not really the product searched for)
- **q** → stop (answers are saved after each product; run the cell again to continue)

Answers go to `data/labels/hand_labels.csv`.

In [8]:
labels = load_labels()
done = set(labels["product_key"])
to_label = sample_for_labelling(p["products"], p["hits"], classification, labelled=set(), per_term=5)
to_label = to_label[~to_label["product_key"].isin(done)]
real_types = p["taxonomy"][p["taxonomy"]["is_negative"] == 0]
print(f"{len(done)} labelled, {len(to_label)} to go")

for i, row in enumerate(to_label.itertuples(), 1):
    options = list(real_types.loc[real_types["search_term"] == row.search_term, "type_id"])
    print(f"\n[{i}/{len(to_label)}] {row.name}  ({row.chain_id}; {row.chain_category if pd.notna(row.chain_category) else '-'})")
    print("  search term:", row.search_term, "| guess:", row.prefilled_type_id)
    print("  " + "  ".join(f"{k}) {t}" for k, t in enumerate(options, 1)) + "  n) no_relevante")
    answer = input("Enter = guess, number, n, q: ").strip().lower()
    if answer == "q":
        break
    if answer == "":
        chosen = row.prefilled_type_id if pd.notna(row.prefilled_type_id) else config.NO_RELEVANTE
    elif answer == "n":
        chosen = config.NO_RELEVANTE
    elif answer.isdigit() and 1 <= int(answer) <= len(options):
        chosen = options[int(answer) - 1]
    else:
        print("  ? not understood, skipped (it will come back next run)")
        continue
    new = pd.DataFrame([{"product_key": row.product_key, "type_id": chosen,
                         "prefilled_type_id": row.prefilled_type_id, "confirmed": 1}])
    new.to_csv(config.LABELS_PATH, mode="a", header=not config.LABELS_PATH.exists() or config.LABELS_PATH.stat().st_size == 0,
               index=False, encoding="utf-8")

1 labelled, 99 to go

[1/99] Chocolate con leche sin azúcar 100 gr  (bm; Chocolate con leche)
  search term: leche | guess: no_relevante
  1) leche_entera  2) leche_semidesnatada  3) leche_desnatada  4) leche_sin_lactosa  5) bebida_vegetal  n) no_relevante



[2/99] Natillas de chocolate Dia Caprichoso 4 x 125 g  (dia; -)
  search term: leche | guess: no_relevante
  1) leche_entera  2) leche_semidesnatada  3) leche_desnatada  4) leche_sin_lactosa  5) bebida_vegetal  n) no_relevante

[3/99] Leche semidesnatada Hacendado  (mercadona; Huevos, leche y mantequilla > Leche y bebidas vegetales > Leche semidesnatada)
  search term: leche | guess: leche_semidesnatada
  1) leche_entera  2) leche_semidesnatada  3) leche_desnatada  4) leche_sin_lactosa  5) bebida_vegetal  n) no_relevante

[4/99] AMBROSI Queso rallado Grana Padano 60 g.  (alcampo; -)
  search term: queso | guess: queso_rallado
  1) queso_curado  2) queso_tierno_fresco  3) queso_lonchas  4) queso_rallado  5) queso_untar  n) no_relevante

[5/99] Aceite de coco virgen Hacendado  (mercadona; Aceite, especias y salsas > Aceite, vinagre y sal > Otros aceites)
  search term: aceite | guess: aceite_oliva_virgen_extra
  1) aceite_oliva_virgen_extra  2) aceite_oliva  3) aceite_girasol  n) no_rel

## Evaluation: embeddings vs TF-IDF

Half of the labels tune τ (best macro-F1) and δ (the margin that best separates right from wrong guesses);
the other half measures accuracy. With ~50 test products the accuracy has a wide 95% interval: we report it.
Copy the tuned τ/δ into `config.THRESHOLDS` and rebuild.

In [9]:
labels = load_labels()
results = [evaluate(p, labels)]
if p["vectorizer"].method == "embedding":
    results.append(evaluate(prepare(force_tfidf=True), labels))

summary = pd.DataFrame([{
    "method": r["method"], "tau": r["tau"], "delta": r["delta"], "n_test": r["report"]["n"],
    "accuracy": r["report"]["accuracy"],
    "accuracy 95% CI": "{:.2f}–{:.2f}".format(*r["report"]["accuracy_ci95"]),
    "macro F1": r["report"]["macro_f1"],
    "relevant precision": r["report"]["relevant_precision"],
    "relevant recall": r["report"]["relevant_recall"],
} for r in results]).round(3)
for r in results:
    for problem in r["problems"]:
        print("label problem:", problem)
summary

,method,tau,delta,n_test,accuracy,accuracy 95% CI,macro F1,relevant precision,relevant recall
0,tfidf,0.16,0.015,49,0.816,0.69–0.90,0.652,0.933,0.903


In [10]:
# Confusion matrix (test half) of the first method
results[0]["report"]["confusion"]

predicted,aceite_oliva,agua_con_gas,atun_aceite_oliva,detergente_capsulas,detergente_delicadas,detergente_liquido,garbanzo_seco,hamburguesas,manzana_roja,no_relevante,...,pollo_pechuga,queso_lonchas,queso_tierno_fresco,salmon_ahumado,tomate_fresco,tomate_frito,tomate_triturado,yogur_griego,yogur_vegetal,zumo_naranja_exprimido
true,,,,,,,,,,,,,,,,,,,,,
aceite_oliva,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
agua_sin_gas,0,2,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
atun_aceite_oliva,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
atun_natural,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
detergente_capsulas,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
detergente_delicadas,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
detergente_liquido,0,0,0,0,0,2,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
garbanzo_seco,0,0,0,0,0,0,2,0,0,0,...,0,0,0,0,0,0,0,0,0,0
hamburguesas,0,0,0,0,0,0,0,3,0,0,...,0,0,0,0,0,0,0,0,0,0


## Similar-product search

`find_similar` returns the products closest to a product (or to any text), optionally in one chain or type.
`compare_types.ipynb` uses it to find each product's equivalent in other chains.

In [11]:
find_similar("leche semidesnatada", k=5, chain="aldi",
             products=p["products"], classification=classification, vectorizer=p["vectorizer"])

,product_key,name,chain_id,type_id,similarity
5254,aldi:600260000,Leche fresca semidesnatada,aldi,leche_semidesnatada,0.880712
5208,aldi:82000,Leche con calcio semidesnatada,aldi,leche_semidesnatada,0.804703
5209,aldi:81900,Leche semidesnatada sin lactosa,aldi,leche_semidesnatada,0.799934
5210,aldi:81000,Leche desnatada,aldi,leche_desnatada,0.727752
5266,aldi:45700,Leche desnatada sin lactosa,aldi,leche_sin_lactosa,0.529549
